# Creating a delta table out of the external federal table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbr_dev.music_analytics.target_artists_cdc (
    artist_id INT,
    artist_name STRING,
    record_label STRING,
    country STRING,
    contract_signed_year INT,
    updated_at TIMESTAMP,
    is_deleted BOOLEAN,
    _synced_at TIMESTAMP
);

INSERT OVERWRITE dbr_dev.music_analytics.target_artists_cdc
SELECT 
    artist_id,
    artist_name,
    record_label,
    country,
    contract_signed_year,
    updated_at,
    is_deleted,
    current_timestamp() AS _synced_at
FROM neon_catalog.public.artists_record_labels;

SELECT * FROM dbr_dev.music_analytics.target_artists_cdc;

# Merging the source data changes into the target delta table

In [0]:
%sql
MERGE INTO dbr_dev.music_analytics.target_artists_cdc AS target
USING neon_catalog.public.artists_record_labels AS source
ON target.artist_id = source.artist_id

WHEN MATCHED AND source.is_deleted = TRUE THEN
  UPDATE SET 
    target.is_deleted = TRUE,
    target.updated_at = source.updated_at,
    target._synced_at = current_timestamp()

WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET 
    target.artist_name = source.artist_name,
    target.record_label = source.record_label,
    target.country = source.country,
    target.contract_signed_year = source.contract_signed_year,
    target.updated_at = source.updated_at,
    target.is_deleted = source.is_deleted,
    target._synced_at = current_timestamp()


WHEN NOT MATCHED AND source.is_deleted = FALSE THEN
  INSERT (
    artist_id,
    artist_name,
    record_label,
    country,
    contract_signed_year,
    updated_at,
    is_deleted,
    _synced_at
  )
  VALUES (
    source.artist_id,
    source.artist_name,
    source.record_label,
    source.country,
    source.contract_signed_year,
    source.updated_at,
    source.is_deleted,
    current_timestamp()
  );

# Making sure the updates are reflected

In [0]:
%sql
SELECT 
    artist_id, 
    artist_name, 
    record_label, 
    is_deleted, 
    updated_at, 
    _synced_at
FROM dbr_dev.music_analytics.target_artists_cdc
ORDER BY artist_id;